# Dịch vụ RAG pháp luật có trace, load test, SLO và chi phí (portfolio)

Ghép mọi thứ của module thành một dịch vụ nhỏ mà bạn vận hành được: hỏi đáp trên Zalo Legal (retrieval dense `bge-m3`,
sinh bằng `qwen2.5:3b` qua Ollama hoặc model GGUF qua llama.cpp server), mỗi request một trace OpenTelemetry ghi ra tệp
JSON Lines, load test open loop ở vài mức tải, rồi một **báo cáo vận hành**: p50/p95/p99 có khoảng tin cậy theo từng bước,
SLO và burn rate, chi phí mỗi 1 000 request (self-host và API), tác dụng của prompt caching, và khuyến nghị capacity.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm đọc trace. Test chấm chúng trên một tệp trace nhỏ tự dựng.
- Cell `explore` là phần thiết kế của bạn. Dùng lại lời giải của bài Basic 01, 03 và Intermediate 02 (dán vào notebook).
- Cần Ollama với `qwen2.5:3b` và `bge-m3`, bộ `zalo-legal` và cache `bge-m3.npy`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/20-evaluation-production/exercises/advanced/02-production-service -q`

In [ ]:
# explore: đưa gốc repo vào sys.path, tìm thư mục dữ liệu
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
ZALO = ROOT / "data" / "raw" / "zalo-legal"

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import math

import numpy as np
import pandas as pd

## 1. Đọc trace

Trace là tệp JSON Lines theo đúng định dạng của `JsonlSpanExporter` (bài Basic 03): mỗi dòng có `name`, `trace_id`,
`span_id`, `parent_span_id`, `start_ns`, `end_ns`, `duration_ms`, `status`, `service`, `attributes`.

`load_spans(path) -> list[dict]`: đọc mọi dòng không rỗng.

`span_table(spans) -> pd.DataFrame`: một dòng mỗi tên span, sắp theo tên, cột đúng thứ tự `SPAN_COLUMNS`: `name`,
`count`, `error_rate` (tỉ lệ `status == "ERROR"`), `p50_ms`, `p95_ms`, `p99_ms` (`np.percentile` của `duration_ms`, mọi
span kể cả lỗi), `share_of_root` (tổng `duration_ms` của tên đó chia tổng `duration_ms` của các span gốc, tức span có
`parent_span_id` là `null`).

`cost_per_1k(spans) -> float`: trung bình thuộc tính `app.cost_usd` của các span gốc (span gốc thiếu thuộc tính này tính
là 0) nhân 1 000. Không có span gốc: `ValueError`.

`root_slo(spans, threshold_ms, target) -> dict`: trên các span gốc, request **tốt** khi status không phải `"ERROR"` và
`duration_ms <= threshold_ms`. Trả `n`, `good_ratio`, `burn_rate` (= tỉ lệ xấu chia `1 - target`), `met`. Không có span
gốc: `ValueError`.

In [ ]:
SPAN_COLUMNS = ["name", "count", "error_rate", "p50_ms", "p95_ms", "p99_ms", "share_of_root"]


def load_spans(path) -> list[dict]:
    # TODO
    raise NotImplementedError


def span_table(spans: list[dict]) -> pd.DataFrame:
    # TODO
    raise NotImplementedError


def cost_per_1k(spans: list[dict]) -> float:
    # TODO
    raise NotImplementedError


def root_slo(spans: list[dict], threshold_ms: float, target: float) -> dict:
    # TODO
    raise NotImplementedError

## 2. Thiết kế của bạn

Gợi ý các bước (thứ tự và cách làm là của bạn):

1. **Dịch vụ.** Một hàm `answer(question)` gồm các bước có span riêng: guardrail input (che PII, module 19), retrieval
   (embed câu hỏi bằng `bge-m3`, top-3 trên cache), dựng prompt (system cố định **trước**, tài liệu và câu hỏi sau), gọi LLM
   **stream** để đo TTFT (ghi `gen_ai.usage.*`, TTFT, model vào span), guardrail output. Exporter JSON Lines.
2. **Tải.** Câu hỏi lấy vòng từ 300 câu ghim. Đo thời gian phục vụ một request trước, ước công suất, rồi chạy open loop
   (bài Intermediate 02) ở 3 mức tải từ thấp tới khoảng 80% công suất, mỗi mức ít nhất 30 request. Ollama dùng chung với
   người khác thì đừng vượt công suất: hàng đợi của bạn là độ trễ của họ.
3. **Báo cáo số.** Bảng `span_table` theo mức tải; p50/p95/p99 E2E có khoảng tin cậy bootstrap (bài Basic 01); TTFT; SLO
   bạn đề xuất và burn rate ở từng mức; chi phí mỗi 1 000 request: (a) self-host với một giá GPU **giả định** ghi rõ, chia
   cho thông lượng ở mức SLO, (b) nếu chạy cùng token qua 2 model API theo bảng giá có ngày kiểm (module 11 mục 3), có và
   không có prompt caching.
4. **Prompt caching.** Đo TTFT khi phần cố định đứng trước so với khi một thứ đổi theo request (ví dụ ngày giờ) đứng đầu
   system prompt.
5. Viết `REPORT.md` theo khung, làm trắc nghiệm của bài trong app.

In [ ]:
# explore: thiết kế của bạn ở đây